
# Adobe Analytics Reporting API — Reproducibility & Isolation Tests

A public-safe notebook for investigating cases where the **same historical date returns different metric values under different report request contexts**.

This version intentionally contains **no production report-suite IDs, segment IDs, calculated-metric IDs, organization identifiers, credentials, stakeholder data, or real output values**.

## What this notebook demonstrates

1. **Identical-request stability** — does the same payload return the same result repeatedly?
2. **Date-range window isolation** — does the result for one target date change when only the enclosing report window changes?
3. **Segment-context isolation** — does the effect persist when segment filters are removed or tested individually?

The goal is not to infer an internal platform root cause from the client side. The goal is to build a **controlled, repeatable isolation test** that separates request-context effects from random API fluctuation.



## 1. Configuration

Keep credentials and production identifiers outside the notebook. Set these environment variables locally before running:

- `ADOBE_GLOBAL_COMPANY_ID`
- `ADOBE_RSID`
- `ADOBE_API_KEY`
- `ADOBE_ACCESS_TOKEN`

Optional segment IDs can be supplied separately when running the segment-isolation section.


In [ ]:

import hashlib
import json
import os
import time
from datetime import datetime, timezone

import pandas as pd
import requests

COMPANY_ID = os.getenv("ADOBE_GLOBAL_COMPANY_ID")
RSID = os.getenv("ADOBE_RSID")
API_KEY = os.getenv("ADOBE_API_KEY")
ACCESS_TOKEN = os.getenv("ADOBE_ACCESS_TOKEN")


def require_config():
    missing = [
        name
        for name, value in {
            "ADOBE_GLOBAL_COMPANY_ID": COMPANY_ID,
            "ADOBE_RSID": RSID,
            "ADOBE_API_KEY": API_KEY,
            "ADOBE_ACCESS_TOKEN": ACCESS_TOKEN,
        }.items()
        if not value
    ]
    if missing:
        raise RuntimeError(
            "Missing environment variables: " + ", ".join(missing)
        )


def build_headers():
    require_config()
    return {
        "Authorization": f"Bearer {ACCESS_TOKEN}",
        "x-api-key": API_KEY,
        "x-proxy-global-company-id": COMPANY_ID,
        "Accept": "application/json",
        "Content-Type": "application/json",
    }


def report_url():
    require_config()
    return f"https://analytics.adobe.io/api/{COMPANY_ID}/reports"


## 2. Reusable request and validation helpers

In [ ]:

DEFAULT_SETTINGS = {
    "countRepeatInstances": True,
    "includeAnnotations": False,
    "limit": 100,
    "page": 0,
    "dimensionSort": "asc",
    "nonesBehavior": "exclude-nones",
}


def metric_label(metric_id: str) -> str:
    """Convert an API metric ID into a compact DataFrame column label."""
    return metric_id.split("/")[-1].replace("-", "_")


def build_payload(
    date_range: str,
    metrics=("metrics/orders", "metrics/visits"),
    segments=(),
    dimension="variables/daterangeday",
    rsid=None,
):
    """Build a minimal Adobe Analytics report request."""
    active_rsid = rsid or RSID
    if not active_rsid:
        raise RuntimeError("RSID is not configured.")

    global_filters = [
        {"type": "segment", "segmentId": segment_id}
        for segment_id in segments
        if segment_id
    ]
    global_filters.append({"type": "dateRange", "dateRange": date_range})

    return {
        "rsid": active_rsid,
        "globalFilters": global_filters,
        "metricContainer": {
            "metrics": [
                {"columnId": str(i), "id": metric_id}
                for i, metric_id in enumerate(metrics)
            ]
        },
        "dimension": dimension,
        "settings": DEFAULT_SETTINGS.copy(),
    }


def payload_hash(payload: dict) -> str:
    """Hash the canonicalized request so repeated cases can be verified."""
    canonical = json.dumps(payload, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()


def post_report(payload: dict, timeout=120) -> dict:
    """Execute a report request and reject partial/failed responses."""
    response = requests.post(
        report_url(),
        headers=build_headers(),
        json=payload,
        timeout=timeout,
    )

    # A partial response should not be treated as a valid comparison point.
    if response.status_code != 200:
        raise RuntimeError(
            f"Unexpected HTTP {response.status_code}: {response.text[:500]}"
        )

    return response.json()


def response_to_frame(result: dict, metric_ids) -> pd.DataFrame:
    """Normalize report rows into a date-indexed DataFrame."""
    labels = [metric_label(metric_id) for metric_id in metric_ids]
    records = []

    for row in result.get("rows", []):
        values = row.get("data", [])
        record = {"date": row.get("value")}
        for i, label in enumerate(labels):
            record[label] = values[i] if i < len(values) else None
        records.append(record)

    frame = pd.DataFrame(records)
    if frame.empty:
        raise RuntimeError("The report returned no rows.")

    frame["date_parsed"] = pd.to_datetime(frame["date"], errors="coerce")
    for label in labels:
        frame[label] = pd.to_numeric(frame[label], errors="coerce")

    return frame.sort_values("date_parsed").reset_index(drop=True)


def run_case(label: str, payload: dict, metric_ids) -> pd.DataFrame:
    started_at = datetime.now(timezone.utc).isoformat()
    result = post_report(payload)
    frame = response_to_frame(result, metric_ids)
    frame["case"] = label
    frame["run_time_utc"] = started_at
    frame["payload_sha256"] = payload_hash(payload)
    return frame


def extract_target_row(frame: pd.DataFrame, target_date: str) -> pd.Series:
    """Return exactly one row matching a YYYY-MM-DD target date."""
    target = pd.Timestamp(target_date).normalize()
    matched = frame.loc[frame["date_parsed"].dt.normalize() == target]

    if len(matched) != 1:
        raise RuntimeError(
            f"Expected exactly one row for {target_date}, found {len(matched)}."
        )

    return matched.iloc[0]



## 3. Test A — identical-request stability

Run **the exact same payload** multiple times. If the payload hash is identical but returned values differ, the instability is not explained by a client-side request change.

If all repeats are stable, move on to controlled context changes.


In [ ]:

METRICS = ("metrics/orders", "metrics/visits")

# Replace these example dates with a non-sensitive test window from your environment.
BASE_RANGE = "2026-01-10T00:00:00.000/2026-01-17T00:00:00.000"
BASE_SEGMENTS = ()


def identical_request_stability(
    date_range=BASE_RANGE,
    metrics=METRICS,
    segments=BASE_SEGMENTS,
    runs=3,
    pause_seconds=1,
):
    payload = build_payload(
        date_range=date_range,
        metrics=metrics,
        segments=segments,
    )

    frames = []
    for run_no in range(1, runs + 1):
        frame = run_case(f"run_{run_no}", payload, metrics)
        frame["run"] = run_no
        frames.append(frame)
        if run_no < runs:
            time.sleep(pause_seconds)

    combined = pd.concat(frames, ignore_index=True)
    metric_cols = [metric_label(metric_id) for metric_id in metrics]

    summary_rows = []
    for date, group in combined.groupby("date", dropna=False):
        row = {"date": date}
        for metric in metric_cols:
            values = group[metric].dropna()
            row[f"{metric}_min"] = values.min() if not values.empty else None
            row[f"{metric}_max"] = values.max() if not values.empty else None
            row[f"{metric}_stable"] = (
                values.nunique(dropna=False) <= 1 if not values.empty else False
            )
        summary_rows.append(row)

    summary = pd.DataFrame(summary_rows)
    return combined, summary


# Example:
# identical_runs, identical_summary = identical_request_stability()
# display(identical_summary)



## 4. Test B — date-range window isolation

Hold the **report suite, segments, metrics, dimension, start point, and target historical date** constant while changing only the enclosing report window.

The S/A/B pattern below is useful because it distinguishes:

- a single-day query (`S`),
- a wider window ending at boundary A (`A`), and
- the same wider window with only the end boundary moved (`B`).

Each case is repeated in an alternating sequence so that a stable case-specific difference is distinguishable from random per-call fluctuation.


In [ ]:

TARGET_DATE = "2026-01-12"

RANGES = {
    "S": "2026-01-12T00:00:00.000/2026-01-13T00:00:00.000",
    "A": "2026-01-10T00:00:00.000/2026-01-20T00:00:00.000",
    "B": "2026-01-10T00:00:00.000/2026-01-21T00:00:00.000",
}


def date_range_window_test(
    target_date=TARGET_DATE,
    ranges=RANGES,
    metrics=METRICS,
    segments=(),
    repeats=3,
    pause_seconds=1,
):
    payloads = {
        case: build_payload(
            date_range=date_range,
            metrics=metrics,
            segments=segments,
        )
        for case, date_range in ranges.items()
    }

    records = []
    for repeat in range(1, repeats + 1):
        for case in ("S", "A", "B"):
            label = f"{case}{repeat}"
            frame = run_case(label, payloads[case], metrics)
            row = extract_target_row(frame, target_date)

            record = {
                "case": case,
                "repeat": repeat,
                "target_date": target_date,
                "payload_sha256": payload_hash(payloads[case]),
            }
            for metric_id in metrics:
                label_name = metric_label(metric_id)
                record[label_name] = row[label_name]
            records.append(record)
            time.sleep(pause_seconds)

    result = pd.DataFrame(records)

    metric_cols = [metric_label(metric_id) for metric_id in metrics]
    stability = (
        result.groupby("case")[metric_cols]
        .nunique(dropna=False)
        .eq(1)
        .rename(columns=lambda c: f"{c}_stable")
    )

    first_values = (
        result.sort_values("repeat")
        .groupby("case", as_index=True)[metric_cols]
        .first()
    )

    comparison = first_values.T
    if {"S", "A", "B"}.issubset(comparison.columns):
        comparison["A_minus_S"] = comparison["A"] - comparison["S"]
        comparison["B_minus_S"] = comparison["B"] - comparison["S"]
        comparison["B_minus_A"] = comparison["B"] - comparison["A"]

    return result, stability, comparison


# Example:
# window_runs, window_stability, window_comparison = date_range_window_test()
# display(window_stability)
# display(window_comparison)



## 5. Test C — segment-context isolation

If a date-range effect is reproducible, test whether a specific segment is required for the effect.

The cleanest baseline is **no request-level segments**, followed by one-segment-at-a-time cases. This does not prove that a segment has zero influence; it tests whether the segment is a **required condition for reproduction**.


In [ ]:

# Add non-sensitive labels and supply the actual IDs only through environment variables.
SEGMENT_CASES = {
    "no_request_segments": (),
    "segment_a_only": tuple(
        filter(None, [os.getenv("ADOBE_TEST_SEGMENT_A")])
    ),
    "segment_b_only": tuple(
        filter(None, [os.getenv("ADOBE_TEST_SEGMENT_B")])
    ),
}


def segment_isolation_test(
    target_date=TARGET_DATE,
    range_a=RANGES["A"],
    range_b=RANGES["B"],
    metrics=METRICS,
    segment_cases=SEGMENT_CASES,
    repeats=2,
):
    rows = []

    for context_label, segments in segment_cases.items():
        if context_label != "no_request_segments" and not segments:
            continue

        local_ranges = {"A": range_a, "B": range_b}
        run_result, stability, comparison = date_range_window_test(
            target_date=target_date,
            ranges={
                "S": f"{target_date}T00:00:00.000/"
                     f"{(pd.Timestamp(target_date) + pd.Timedelta(days=1)).date()}T00:00:00.000",
                **local_ranges,
            },
            metrics=metrics,
            segments=segments,
            repeats=repeats,
        )

        metric_cols = [metric_label(metric_id) for metric_id in metrics]
        first_ab = (
            run_result[run_result["case"].isin(["A", "B"])]
            .sort_values("repeat")
            .groupby("case")[metric_cols]
            .first()
        )

        for metric in metric_cols:
            rows.append(
                {
                    "segment_context": context_label,
                    "metric": metric,
                    "A": first_ab.loc["A", metric],
                    "B": first_ab.loc["B", metric],
                    "B_minus_A": (
                        first_ab.loc["B", metric]
                        - first_ab.loc["A", metric]
                    ),
                    "A_stable": bool(stability.loc["A", f"{metric}_stable"]),
                    "B_stable": bool(stability.loc["B", f"{metric}_stable"]),
                }
            )

    return pd.DataFrame(rows)


# Example:
# segment_summary = segment_isolation_test()
# display(segment_summary)



## 6. Interpretation framework

A useful evidence pattern is:

- **Identical payload is stable** across repeated calls.
- **A and B are each stable** across repeats.
- The target historical date returns **different values between A and B** when only the report end boundary changes.
- The same pattern persists in the **no-request-segment baseline**.

That supports a conclusion of **deterministic request-context dependency** rather than simple random API fluctuation.

It does **not**, by itself, establish the platform's internal root cause, cache behavior, or whether the behavior is expected. Those require platform documentation, support confirmation, or an independent processing-path comparison.

### Public repository hygiene

Before committing a real investigation notebook, remove or externalize:

- organization/company IDs and report-suite IDs,
- VRS IDs and parent report-suite IDs,
- segment and calculated-metric IDs,
- authentication filenames, tokens, API keys, and local package paths,
- exact production date ranges tied to an incident,
- raw API responses and notebook outputs containing real traffic/order values,
- stakeholder names, internal ticket references, internal URLs, and business-specific comments,
- administrative or mutating API calls that are unrelated to the analytical method.
